In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import os
pd.options.mode.chained_assignment = None
import optuna
import time
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold, train_test_split
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, RobustScaler
import pickle
from collections import defaultdict

/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [24]:
path = "saved_files/dataset"
depths = ["in_0_1"]#, "in_1_2", "in_2_3", "in_3_4"]

dfs = {}
for archivo in os.listdir(path):
    if archivo.endswith("_features.csv") and any(depth in archivo for depth in depths):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        # quitamos "_features" al final del nombre
        dfs[nombre_sin_extension[:-9]] = pd.read_csv(ruta_completa)



# Limpiamos valores nulos
for nombre_df, df in dfs.items():
    for band_set in ["rhow", "rhown","rtoa"]:
        dfs[nombre_df] = df.dropna()

dfs_to_keep = [
    'C2X-Complex_rhow_9x9_depth_in_0_1', 
    'TOA_15x15_depth_in_0_1',
    'C2X-Complex_rhown_9x9_depth_in_0_1',
    'C2X-Complex_rhow_5x5_depth_in_0_1',
    'C2RCC_rhow_5x5_depth_in_0_1',
    'C2RCC_rhow_15x15_depth_in_0_1', 
    'C2X-Complex_rhown_15x15_depth_in_0_1',
    'C2RCC_rhown_5x5_depth_in_0_1', 
    'C2X-Complex_rhow_15x15_depth_in_0_1',
    'C2RCC_rhow_9x9_depth_in_0_1',
    'C2X-Complex_rhow_5x5_depth_in_1_2', 
    'C2X_rhow_3x3_depth_in_1_2',
    'C2X-Complex_rhown_5x5_depth_in_1_2',
    'C2X-Complex_rhow_9x9_depth_in_1_2',
    'C2X-Complex_rhow_3x3_depth_in_1_2',
    'C2X-Complex_rhown_3x3_depth_in_1_2',
    'C2RCC_rhown_3x3_depth_in_1_2',
    'C2X-Complex_rhown_9x9_depth_in_1_2',
    'C2X-Complex_rhow_15x15_depth_in_1_2', 
    'C2X_rhow_5x5_depth_in_1_2',
    'TOA_15x15_depth_in_2_3',
    'TOA_9x9_depth_in_2_3',
    'TOA_5x5_depth_in_2_3', 
    'C2X-Complex_rhow_5x5_depth_in_2_3',
    'C2RCC_rhown_5x5_depth_in_2_3', 
    'TOA_3x3_depth_in_2_3',
    'C2X-Complex_rhown_5x5_depth_in_2_3', 
    'C2RCC_rhow_3x3_depth_in_2_3',
    'C2X-Complex_rhown_9x9_depth_in_2_3', 
    'C2X_rhow_9x9_depth_in_2_3',
    'TOA_9x9_depth_in_3_4',
    'TOA_3x3_depth_in_3_4',
    'TOA_5x5_depth_in_3_4',
    'C2X-Complex_rhow_5x5_depth_in_3_4', 
    'TOA_1x1_depth_in_3_4',
    'TOA_15x15_depth_in_3_4',
    'C2X-Complex_rhown_5x5_depth_in_3_4',
    'C2X-Complex_rhow_9x9_depth_in_3_4',
    'C2X-Complex_rhow_15x15_depth_in_3_4',
    'C2X-Complex_rhown_9x9_depth_in_3_4'
    ]

dfs = {k: dfs[k] for k in dfs_to_keep if k in dfs}


for nombre_df, df in dfs.items():
    # Marcamos las columnas de CHl alta (equivalente a quantile(0.93))
    df["High_Chl"] = df["Chl"]>5
    # Sacamos la estación de cada fecha
    df['Date'] = pd.to_datetime(df['Date'])
    def get_season(month):
        if month in [12, 1, 2]:
            return 'Invierno'
        elif month in [3, 4, 5]:
            return 'Primavera'
        elif month in [6, 7, 8]:
            return 'Verano'
        else:
            return 'Otoño'
    df['Season'] = df['Date'].dt.month.apply(get_season)
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype('category')
    df = pd.concat([df.drop(columns=["Season"]),pd.get_dummies(df["Season"])], axis=1)
    dfs[nombre_df] = df

In [26]:
# Plantillas base con los parámetros que no se han optimizado con Optuna
base_params = {
    "XGB": {
        'device': 'cpu',
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'enable_categorical': True,
        'eval_metric': 'rmse'
    },
    "LBM": {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'device': 'cpu',
        'verbosity': -1
    },
    "MLP": {
        'max_iter': 200,
        'shuffle': True,
        'tol': 1e-4,
        'n_iter_no_change': 25,
        'verbose': False,
        'early_stopping': True,
        'validation_fraction': 0.2
    },
    "SVR": {
        'kernel': 'rbf',
        'gamma': 'scale',
        'shrinking': True,
        'tol': 1e-3,
        'max_iter': -1,
        'verbose': False
    },
    "KNN": {
        'weights': 'distance',
        'algorithm': 'auto',
        'metric': 'minkowski',
        'p': 2,
        'n_jobs': -1
    },
    "RF": {
        'criterion': 'squared_error',
        'random_state': 42,
        'verbose': 0
    },
    "CAT": {
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'random_seed': 42,
        'early_stopping_rounds': 50,
        'verbose': False
    },
    "ELN": {
        'fit_intercept': True,
        'max_iter': 1000,
        'tol': 1e-4,
        'selection': 'cyclic',
        'random_state': 42
    }
}

In [28]:
with open(f"training_results/selection_results_in_0_1.pkl", "rb") as f:
    results = pickle.load(f)

In [35]:
# Porque se guardaron las keys, no los valores
mlp_hidden_layer_map = {
    "128": (128,),
    "256": (256,),
    "256_128": (256, 128),
    "128_64": (128, 64)
}

model_params = {}
for (dataset, model), data in results.items():
    best_params = data["best_params"].copy()

    # Corrección para MLP
    if model == "MLP" and "hidden_layer_sizes" in best_params:
        key = best_params["hidden_layer_sizes"]
        if isinstance(key, str) and key in mlp_hidden_layer_map:
            best_params["hidden_layer_sizes"] = mlp_hidden_layer_map[key]

    merged = {**base_params.get(model, {}), **best_params}
    if dataset not in model_params:
        model_params[dataset] = {}
    model_params[dataset][model] = merged

In [36]:
model_params

{'C2X-Complex_rhow_9x9_depth_in_0_1': {'XGB': {'device': 'cpu',
   'objective': 'reg:squarederror',
   'tree_method': 'hist',
   'enable_categorical': True,
   'eval_metric': 'rmse',
   'n_estimators': 1000,
   'learning_rate': 0.0402726629694344,
   'max_depth': 7,
   'min_child_weight': 1,
   'subsample': 0.6004268471047662,
   'colsample_bytree': 0.7669438778479738},
  'LBM': {'objective': 'regression',
   'metric': 'rmse',
   'boosting_type': 'gbdt',
   'device': 'cpu',
   'verbosity': -1,
   'learning_rate': 0.02383261306371179,
   'num_leaves': 40,
   'max_depth': 6,
   'min_child_samples': 4,
   'subsample': 0.7800573402452207,
   'colsample_bytree': 0.7172411700707029,
   'n_estimators': 500},
  'MLP': {'max_iter': 200,
   'shuffle': True,
   'tol': 0.0001,
   'n_iter_no_change': 25,
   'verbose': False,
   'early_stopping': True,
   'validation_fraction': 0.2,
   'hidden_layer_sizes': (256, 128),
   'activation': 'tanh',
   'solver': 'sgd',
   'alpha': 4.52527682354138e-05,
  

In [31]:
model_params["C2RCC_rhow_15x15_depth_in_0_1"]["MLP"]

{'max_iter': 200,
 'shuffle': True,
 'tol': 0.0001,
 'n_iter_no_change': 25,
 'verbose': False,
 'early_stopping': True,
 'validation_fraction': 0.2,
 'hidden_layer_sizes': '256_128',
 'activation': 'relu',
 'solver': 'adam',
 'alpha': 0.00040008329958286294,
 'learning_rate': 'adaptive',
 'learning_rate_init': 0.009624147801327707}

In [53]:
def cross_validation_training(dfs, depth):

    FOLDS = 5
    # Para hacer clip a las predicciones y forzar > 0
    correct = True
    results = {}
    rs = 13

    for nombre_df, df in list(dfs.items()):
    #for nombre_df, df in islice(dfs.items(), 3):
        print(f"\n=== Procesando {nombre_df} ===")
        # Ignoramos las columnas de Date, Lat, Lon y Buoy
        df = df.iloc[:, 4:]

        # Separamos el conjunto de datos en train y test: Train 75% Test 25%
        train, test = train_test_split(df, test_size=0.25, random_state=rs, stratify=df["High_Chl"])
        # Seleccionamos la columna que queremos predecir
        target = "Chl"

        # Quitamos esa columna y el indicador de clorofila alta
        X = train.drop(columns=[target, "High_Chl", "Turbidez"])
        # Para y cogemos solamente Chl
        y = train[target]
        # Para poder hacer StratifiedKFold y tener el mismo número de valores de Chl alta en cada fold
        y_class = train["High_Chl"]

        # Definimos X e y para test
        X_test = test.drop(columns=[target, "High_Chl", "Turbidez"])
        y_test = test[target]

        #----Definir modelos usando los params específicos de este dataset----# Esto es diferente a como están en Entrenamiento_V3
        models = {
            "XGB": XGBRegressor(**model_params[nombre_df]["XGB"]),
            "LBM": LGBMRegressor(**model_params[nombre_df]["LBM"]),
            "MLP": MLPRegressor(**model_params[nombre_df]["MLP"]),
            "SVR": SVR(**model_params[nombre_df]["SVR"]),
            "KNN": KNeighborsRegressor(**model_params[nombre_df]["KNN"]),
            "LR": LinearRegression(),
            "RF": RandomForestRegressor(**model_params[nombre_df]["RF"]),
            "CAT": CatBoostRegressor(**model_params[nombre_df]["CAT"]),
            "ELN": ElasticNet(**model_params[nombre_df]["ELN"])
        }


        # Dicts para guardar las predicciones sobre los conjuntos de validación, las y's correspondientes y los índices que corresponden dentro del loop de folds para el ensemble
        val_preds = {name: np.zeros(len(train)) for name in models}
        y_vals = defaultdict(list)
        val_indices = {}
        # Dict para guardar las predicciones sobre test
        test_preds = {name: np.zeros(len(test)) for name in models}
        
        # Dict para guardar resultados
        results[nombre_df] = {name: {'RMSE': [], 'R2': []} for name in models}
        # Stratified KFold de 5 folds
        skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

        # Loop para entrenar cada uno de los modelos
        for name, model in models.items():
            print(f"\n=== Training {name} ===")
            # Loop de folds, manteniendo la proporción de clases (Chl > 5) con y_class
            for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
                print(f"Fold {fold+1}")
                X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
                y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

                # Para modelos basados en distancias escalamos los datos
                if name in ["MLP", "SVR", "KNN", "LR", "ELN"]:
                    # Escalado dentro del loop de folds para evitar data leakage entre folds
                    scaler_X = RobustScaler()
                    scaler_y = RobustScaler()
                    X_train_scaled = scaler_X.fit_transform(X_train)
                    X_val_scaled = scaler_X.transform(X_val)
                    X_test_scaled = scaler_X.transform(X_test)
                    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
                    # Entrenamos modelo con datos escalados
                    model.fit(X_train_scaled, y_train_scaled)
                    # Predicción sobre val y test, haciendo la transformada inversa para devolver y a su escala
                    val_pred = scaler_y.inverse_transform(model.predict(X_val_scaled).reshape(-1, 1)).ravel()
                    test_pred = scaler_y.inverse_transform(model.predict(X_test_scaled).reshape(-1, 1)).ravel()
                # Para modelos basados en árboles no es necesario escalar
                else:
                    # Entrenamos el modelo
                    model.fit(X_train, y_train)
                    # Predicción sobre val y test
                    val_pred = model.predict(X_val)
                    test_pred = model.predict(X_test)

                if correct:
                    val_pred = np.clip(val_pred, 0.3, None)
                    test_pred = np.clip(test_pred, 0.3, None)

                # Guardamos las predicciones sobre val, las y's que les corresponden y los índices
                val_preds[name][val_idx] = val_pred
                if name == list(models.keys())[0]:
                    # Solo lo guardamos una vez
                    y_vals[fold] = y_val
                    val_indices[fold] = val_idx  # val_idx es un array de índices relativos a train

                # Guardamos la predicción de test, haciendo la media entre los folds
                test_preds[name] += test_pred / FOLDS

                # Calculamos y guardamos métricas
                rmse = np.sqrt(mean_squared_error(y_val, val_pred))
                r2 = r2_score(y_val, val_pred)
                results[nombre_df][name]['RMSE'].append(rmse)
                results[nombre_df][name]['R2'].append(r2)

        # Extendemos el dict de resultados con el ensemble
        results[nombre_df]["ENS"] = {'RMSE': [], 'R2': []}

        for fold in range(FOLDS):
            # Índices y valores del fold actual
            fold_val_idx = val_indices[fold]
            meta_X_val = np.vstack([val_preds[model][fold_val_idx] for model in models]).T
            meta_y_val = y_vals[fold]

            # Índices de entrenamiento: todos menos el fold actual
            train_folds = [i for i in range(FOLDS) if i != fold]
            train_idx = np.concatenate([val_indices[i] for i in train_folds])
            meta_X_train = np.vstack([val_preds[model][train_idx] for model in models]).T
            meta_y_train = y.iloc[train_idx]

            # Entrenamos el meta-modelo solo con los otros 4 folds
            meta_model = Ridge().fit(meta_X_train, meta_y_train)

            # Predicción en el fold actual (no visto)
            ensemble_pred = meta_model.predict(meta_X_val)
            if correct:
                ensemble_pred = np.clip(ensemble_pred, 0.3, None)

            rmse = np.sqrt(mean_squared_error(meta_y_val, ensemble_pred))
            r2 = r2_score(meta_y_val, ensemble_pred)
            results[nombre_df]["ENS"]['RMSE'].append(rmse)
            results[nombre_df]["ENS"]['R2'].append(r2)



    # === Evaluación final sobre test ===
        for name in models:
            rmse_test = np.sqrt(mean_squared_error(y_test, test_preds[name]))
            r2_test = r2_score(y_test, test_preds[name])
            results[nombre_df][name]["RMSE test"] = rmse_test
            results[nombre_df][name]["R2 test"] = r2_test

        # Construcción del meta-modelo sobre todo el conjunto de validación
        final_meta_X = np.vstack([val_preds[model] for model in models]).T
        final_meta_y = y.values
        ensemble_model = Ridge().fit(final_meta_X, final_meta_y)

        # Predicción sobre test del ensemble
        meta_X_test = np.vstack([test_preds[model] for model in models]).T
        ensemble_test_pred = ensemble_model.predict(meta_X_test)
        if correct:
            ensemble_test_pred = np.clip(ensemble_test_pred, 0.3, None)
        # Evaluación del ensemble sobre test
        rmse_ens_test = np.sqrt(mean_squared_error(y_test, ensemble_test_pred))
        r2_ens_test = r2_score(y_test, ensemble_test_pred)
        results[nombre_df]["ENS"]["RMSE test"] = rmse_ens_test
        results[nombre_df]["ENS"]["R2 test"] = r2_ens_test

    with open(f"training_results/results_entrenamiento_final_{depth}_rs13.pkl", "wb") as f:
        pickle.dump(results, f)

    return results

In [27]:
depths[0]

'in_0_1'

In [55]:
results = cross_validation_training(dfs, depths[0])


=== Procesando C2X-Complex_rhow_9x9_depth_in_0_1 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando TOA_15x15_depth_in_0_1 ===

=== Training XGB ===
Fold 1


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.085e+02, tolerance: 4.943e-01
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.578e+02, tolerance: 3.834e-01
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations

Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2
Fold 3
Fold 4


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 5

=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando C2X-Complex_rhown_9x9_depth_in_0_1 ===

=== Training XGB ===
Fold 1


...

/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.956e+02, tolerance: 4.943e-01
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.586e+02, tolerance: 3.834e-01
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations

Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando C2RCC_rhow_9x9_depth_in_0_1 ===

=== Training XGB ===
Fold 1


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.405e+02, tolerance: 3.669e-01
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.773e+02, tolerance: 3.557e-01
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations

Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.277e+02, tolerance: 4.943e-01
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.071e+02, tolerance: 3.834e-01
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations

In [56]:
rows = []

for df_name, model_scores in results.items():
    row = {}
    for model_name, metrics in model_scores.items():
        for metric_name, values in metrics.items():
            if isinstance(values, list):  # Solo para los que tienen listas (folds)
                mean_val = np.mean(values)
                std_val = np.std(values)
                row[(metric_name, model_name)] = f"{mean_val:.2f} ± {std_val:.2f}"
            else:
                # Para el ensemble que tiene un único valor
                row[(metric_name, model_name)] = f"{values:.2f}"
    rows.append((df_name, row))

df_results = pd.DataFrame.from_dict(dict(rows), orient="index")
df_results.columns = pd.MultiIndex.from_tuples(df_results.columns, names=["Metric", "Model"])
df_results = df_results.sort_index(axis=1, level=0)
df_results = df_results.sort_index(axis=0)

In [57]:
df_sorted = df_results["R2 test"].copy()
# Añadir una columna auxiliar con el R2 máximo por fila
df_sorted["max_R2"] = df_sorted.max(axis=1)
# Ordenar por esa columna en orden descendente
df_sorted = df_sorted.sort_values("max_R2", ascending=False)
# Eliminar la columna auxiliar
df_sorted = df_sorted.drop(columns="max_R2")
df_sorted

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2X-Complex_rhow_15x15_depth_in_0_1,0.80,0.56,0.78,0.63,0.68,-0.20,0.82,0.64,0.81,0.64
C2X-Complex_rhown_15x15_depth_in_0_1,0.71,0.53,0.78,0.66,0.50,0.47,0.69,0.49,0.80,0.49
C2RCC_rhow_15x15_depth_in_0_1,0.79,0.47,0.76,0.74,0.68,0.36,0.74,0.52,0.76,0.66
C2X-Complex_rhow_5x5_depth_in_0_1,0.38,-0.38,0.52,-0.03,0.19,-3.48,0.64,0.26,0.74,0.35
C2RCC_rhow_5x5_depth_in_0_1,0.73,0.45,0.69,0.71,0.65,0.22,0.69,0.36,0.60,0.73
C2RCC_rhown_5x5_depth_in_0_1,0.68,0.42,0.63,0.72,0.64,0.22,0.66,0.56,0.61,0.66
C2X-Complex_rhow_9x9_depth_in_0_1,0.71,0.27,0.72,0.26,0.56,-0.05,0.72,0.49,0.70,0.62
TOA_15x15_depth_in_0_1,0.57,0.49,0.66,0.71,0.35,0.28,0.36,0.08,0.64,0.26
C2RCC_rhow_9x9_depth_in_0_1,0.65,0.42,0.63,0.68,0.65,0.20,0.61,0.39,0.63,0.61
C2X-Complex_rhown_9x9_depth_in_0_1,0.49,0.24,0.35,0.05,0.34,0.17,0.56,0.51,0.56,0.33


In [42]:
results

{'C2X-Complex_rhow_9x9_depth_in_0_1': {'XGB': {'RMSE': [2.0034609000315937,
    2.507542605891783,
    2.2221568494502772,
    2.1288518613606864,
    1.4320144445293634],
   'R2': [0.6416045436669361,
    0.5174512943273623,
    0.7264351333093318,
    0.6956737314532637,
    0.7953946117480192],
   'RMSE test': 1.3137879351741801,
   'R2 test': 0.9063647499562524},
  'LBM': {'RMSE': [2.060471518457993,
    2.587306016141012,
    2.47152857503235,
    2.199136065117136,
    1.7891223724858138],
   'R2': [0.6209172832141514,
    0.48626386981099934,
    0.6615907908233791,
    0.6752473081597201,
    0.6806239375356997],
   'RMSE test': 1.4277918423099873,
   'R2 test': 0.8894092945252461},
  'MLP': {'RMSE': [2.1380858237750133,
    1.9108225569993218,
    2.647201846090131,
    2.483584591063606,
    1.4009614824133325],
   'R2': [0.591820658226115,
    0.7197889663669637,
    0.6117736444116935,
    0.5858034557318383,
    0.8041720581641995],
   'RMSE test': 1.6466245029425943,
   '